In [1]:

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Audio
import lightning as L
import sys
from lightning.pytorch.callbacks import LearningRateMonitor
sys.path.append('../')
sys.path.append('./')
from importlib import reload
import yaml
import torch

torch.set_float32_matmul_precision('medium')


In [2]:
import lightning_scripts.lightning_ssl_matched_speech_in_noise as lightning 
reload(lightning)
import lightning_scripts.whisper_encoder_arch as whisper_encoder_arch
reload(whisper_encoder_arch)

LitAudioSSL = lightning.LitAudioSSL

## init config. Will be yaml eventually, but start as dict 
config_path = "model_configs/whisper_tiny_barlow_equivariant_lmbda_1e-2_lr_2e-1_eq_lmbda_0e-01.yaml"
config = yaml.load(open(config_path, 'r'), Loader=yaml.FullLoader)

config['num_workers'] = 4
config['hparas']['batch_size'] = 256    
config['hparas']['global_batch_size'] = 256
config['num_gpus'] = 1 

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = LitAudioSSL(config).to(device)


In [3]:
model

LitAudioSSL(
  (audio_rep): AudioToAudioRepresentation(
    (rep): AudioToCochleagram(
      (envelope_extraction): HilbertEnvelopeExtraction()
      (downsampling_op): SincWithKaiserWindow()
      (Cochleagram): Cochleagram(
        (compute_subbands): ComputeSubbands()
        (envelope_extraction): HilbertEnvelopeExtraction()
        (downsampling): SincWithKaiserWindow()
      )
    )
    (compression): ClippedGradPower(
      (compression_function): ClippedGradPowerCompression()
    )
  )
  (model): ModelWithFrontEnd(
    (front_end): AudioToAudioRepresentation(
      (rep): AudioToCochleagram(
        (envelope_extraction): HilbertEnvelopeExtraction()
        (downsampling_op): SincWithKaiserWindow()
        (Cochleagram): Cochleagram(
          (compute_subbands): ComputeSubbands()
          (envelope_extraction): HilbertEnvelopeExtraction()
          (downsampling): SincWithKaiserWindow()
        )
      )
      (compression): ClippedGradPower(
        (compression_function): C

In [4]:
trainer = L.Trainer(
                    # callbacks=[lr_monitor],
                    # limit_train_batches=5,
                    limit_val_batches=2,
                    max_epochs=5,
                    # callbacks=callbacks,
                    #  strategy='ddp_notebook',
                    #  reload_dataloaders_every_n_epochs=-1,
                    devices=1)

trainer.fit(model)

/mnt/home/igriffith/envs/cochdnn_ssl_pl/lib/python3.12/site-packages/lightning/fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /mnt/home/igriffith/envs/cochdnn_ssl_pl/lib/python3. ...
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/mnt/home/igriffith/envs/cochdnn_ssl_pl/lib/python3.12/site-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(

  | Name            | Type                       | Params | Mode 
-----------------------------------------------------------------------
0 | audio_rep       | AudioToAudioRepresentation | 0      | train
1 | model           | ModelWithFrontEnd          | 215 M  

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Rank 0 N training batches 79


For this stretch factor, the stretch effect has better performance.
For this stretch factor, the stretch effect has better performance.
For this stretch factor, the stretch effect has better performance.
For this stretch factor, the stretch effect has better performance.
For this stretch factor, the stretch effect has better performance.
For this stretch factor, the stretch effect has better performance.
For this stretch factor, the stretch effect has better performance.
For this stretch factor, the stretch effect has better performance.
For this stretch factor, the stretch effect has better performance.
For this stretch factor, the stretch effect has better performance.
For this stretch factor, the stretch effect has better performance.
For this stretch factor, the stretch effect has better performance.
For this stretch factor, the stretch effect has better performance.
For this stretch factor, the stretch effect has better performance.
For this stretch factor, the stretch effect has 

OutOfMemoryError: CUDA out of memory. Tried to allocate 16.10 GiB. GPU 0 has a total capacity of 39.56 GiB of which 13.91 GiB is free. Including non-PyTorch memory, this process has 25.64 GiB memory in use. Of the allocated memory 25.14 GiB is allocated by PyTorch, and 13.35 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)